# The best model we'll do
* bangbang



## 0. Setup


In [0]:
NOM_EQUIPE = "Télécacaton"   # ← remplacez par le nom de votre équipe

# Ne touchez pas au reste
TABLE_PREDICTIONS = f"workspace.default.predictions_equipe_{NOM_EQUIPE}"
print(f"Votre table de prédictions : {TABLE_PREDICTIONS}")

## 00. First try with code given

on regarde les données

In [0]:
display(spark.table("workspace.default.histo_ventes_train").limit(10))

On regarde les stats

In [0]:
# Statistiques générales sur le jeu d'entraînement
from pyspark.sql import functions as F

train_df = spark.table("workspace.default.histo_ventes_train")

print("=== Statistiques histo_ventes_train ===")
print(f"Nombre de lignes     : {train_df.count():,}")
print(f"Semaine min          : {train_df.agg(F.min('semaine')).collect()[0][0]}")
print(f"Semaine max          : {train_df.agg(F.max('semaine')).collect()[0][0]}")
print(f"Nombre d'agences     : {train_df.select('code_agence').distinct().count():,}")
print(f"Nombre d'articles    : {train_df.select('code_article').distinct().count():,}")

nb_total = train_df.count()
nb_zeros = train_df.filter(F.col("quantite") == 0).count()
print(f"Lignes avec quantité = 0 : {nb_zeros:,} ({100*nb_zeros/nb_total:.1f}%)")

### Table de test — `histo_ventes_test`

Les lignes que **vous devez prédire**. Même structure que le train, mais `quantite` est vide.

> Il y a **272 344 lignes** à prédire. Vous devez toutes les couvrir.
> Une ligne manquante est comptée comme une prédiction de **0**.

In [0]:
test_df = spark.table("workspace.default.histo_ventes_test")
display(test_df.limit(10))

In [0]:
print(f"Lignes à prédire : {test_df.count():,}")
print(f"Semaine min : {test_df.agg(F.min('semaine')).collect()[0][0]}")
print(f"Semaine max : {test_df.agg(F.max('semaine')).collect()[0][0]}")

### Tables de référence

Ces tables peuvent enrichir vos features.

| Table | Contenu |
|-------|---------|
| `donnees_agence` | Caractéristiques des agences (région, taille, localisation…) |
| `donnees_articles` | Caractéristiques des articles (famille, catégorie, saisonnalité…) |
| `donnees_facturation` | Données de facturation (prix, montants, comportements d'achat…) |

In [0]:
display(spark.table("workspace.default.donnees_agence").limit(5))

In [0]:
display(spark.table("workspace.default.donnees_articles").limit(10))

In [0]:
display(spark.table("workspace.default.donnees_facturation").limit(5))

### Étape 3 — Construire une baseline

Pour démarrer rapidement, voici une **baseline saisonnière N-1** : pour chaque ligne à prédire, on prend la quantité vendue la même semaine l'année précédente. Si aucune donnée N-1 n'existe, on prédit 0.

C'est simple mais souvent difficile à battre sur des données saisonnières !

> **Scores de référence à battre :**
> - Baseline saisonnière simple → WAPE ≈ 1.387
> - Blend pondéré N-1 + moyenne → WAPE ≈ 1.259

Regardez aussi les notebooks **`01_LightGBM_Approche1`** et **`02_XGBoost_Approche2`** pour des approches ML plus avancées.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import LongType

train_df = spark.table("workspace.default.histo_ventes_train")
test_df  = spark.table("workspace.default.histo_ventes_test")

# Construire la clé de jointure N-1 : "2025-27" → "2024-27"
def semaine_n_moins_1(col):
    annee   = F.split(col, "-")[0].cast("int")
    num_sem = F.split(col, "-")[1]
    return F.concat((annee - 1).cast("string"), F.lit("-"), num_sem)

# Historique N-1 : renommer quantite → quantite_n1 et semaine → semaine_n1
train_n1 = (
    train_df
    .select(
        F.col("semaine").alias("semaine_n1"),
        "code_agence",
        "code_article",
        F.col("quantite").alias("quantite_n1"),
    )
)

# Jointure : pour chaque ligne de test, trouver la quantité N-1
predictions_df = (
    test_df
    .withColumn("semaine_n1", semaine_n_moins_1(F.col("semaine")))
    .join(train_n1, on=["semaine_n1", "code_agence", "code_article"], how="left")
    .withColumn("quantite", F.coalesce(F.col("quantite_n1"), F.lit(0)).cast(LongType()))
    .select("semaine", "code_agence", "code_article", "quantite")
)

print(f"Nombre de prédictions générées : {predictions_df.count():,}")
display(predictions_df.limit(10))

### Étape 4 — Auto-évaluer votre WAPE

Les semaines **2025-01 à 2025-26** sont dans `histo_ventes_train` — vous connaissez les vraies valeurs.
Utilisez-les comme **jeu de validation** pour estimer votre WAPE avant de soumettre officiellement.

**Recommandation :** entraînez votre modèle sur les données **avant 2025-01**, validez sur **2025-01 à 2025-26**.

In [0]:
def compute_wape(predictions_df, actuals_df, pred_col="quantite", actual_col="quantite_actuel"):
    """
    Calcule le WAPE entre un DataFrame de prédictions et les vraies valeurs.
    Les deux DataFrames doivent avoir : semaine, code_agence, code_article.
    """
    joined = predictions_df.join(
        actuals_df,
        on=["semaine", "code_agence", "code_article"],
        how="inner",
    )
    row = joined.agg(
        (
            F.sum(F.abs(F.col(pred_col).cast("double") - F.col(actual_col).cast("double")))
            / (F.sum(F.col(actual_col).cast("double")) + F.lit(1e-10))
        ).alias("wape")
    ).collect()[0]
    return float(row["wape"])


# Exemple : calcul du WAPE de validation pour la baseline N-1
# (on génère des prédictions sur les semaines 2025-01 à 2025-26)

train_avant_2025 = train_df.filter(F.col("semaine") < "2025-01")
val_actuel = (
    train_df
    .filter(F.col("semaine").between("2025-01", "2025-26"))
    .select("semaine", "code_agence", "code_article", F.col("quantite").alias("quantite_actuel"))
)

# Construire les prédictions N-1 sur la période de validation
train_n1_val = (
    train_avant_2025
    .select(
        F.col("semaine").alias("semaine_n1"),
        "code_agence",
        "code_article",
        F.col("quantite").alias("quantite_n1"),
    )
)

# Pour chaque ligne de validation, chercher la quantité de la même semaine l'année d'avant
val_preds = (
    val_actuel
    .withColumn("semaine_n1", semaine_n_moins_1(F.col("semaine")))
    .join(train_n1_val, on=["semaine_n1", "code_agence", "code_article"], how="left")
    .withColumn("quantite", F.coalesce(F.col("quantite_n1"), F.lit(0)).cast(LongType()))
    .select("semaine", "code_agence", "code_article", "quantite")
)

wape_val = compute_wape(val_preds, val_actuel)
print(f"WAPE de validation (baseline N-1) : {wape_val:.4f}")
print()
print("Scores de référence :")
print("  Baseline saisonnière  → WAPE ≈ 1.3870")
print("  Blend pondéré         → WAPE ≈ 1.2590")
print()
if wape_val < 1.259:
    print("Vous battez la référence ! Pensez à soumettre officiellement.")
else:
    print("Continuez à améliorer votre modèle avant de soumettre !")

### Étape 5 — Sauvegarder vos prédictions

Le format attendu est simple : **4 colonnes**, une ligne par combinaison à prédire.

| Colonne | Type | Description |
|---------|------|-------------|
| `semaine` | STRING | Format `"YYYY-WW"` — ex : `"2025-27"` |
| `code_agence` | LONG | Identifiant de l'agence |
| `code_article` | LONG | Identifiant de l'article |
| `quantite` | LONG | Votre prédiction (≥ 0) |

> **Couvrez toutes les 272 344 lignes de `histo_ventes_test`.**
> Une ligne manquante compte comme une prédiction de **0**.

In [0]:
# Vérifier que vos prédictions ont le bon format
print(f"Nombre de lignes dans vos prédictions : {predictions_df.count():,}")
print(f"Nombre attendu                         : 272 344")
print()
predictions_df.printSchema()

#### Sauvegarder dans votre table Delta

Exécutez la cellule ci-dessous quand vos prédictions sont prêtes.

> Vous pouvez écraser votre table autant de fois que vous voulez — **chaque soumission officielle est comptabilisée séparément** (5 max) depuis l'application.

In [0]:
# Décommentez et exécutez quand vos prédictions sont prêtes
# predictions_df.write.mode("overwrite").saveAsTable(TABLE_PREDICTIONS)
# print(f"✅ Prédictions sauvegardées dans : {TABLE_PREDICTIONS}")

#### Si vous êtes plusieurs dans l'équipe

La soumission est protégée : **seule la personne qui a créé la table peut soumettre** par défaut (Unity Catalog n'accorde l'accès MODIFY qu'au créateur de la table).

Si vous voulez qu'un coéquipier puisse également soumettre, le créateur de la table doit lui donner accès :

```sql
GRANT MODIFY ON TABLE workspace.default.predictions_equipe_<NOM_EQUIPE>
TO 'email.coequipier@centrale-supelec.fr';
```

Cela garantit aussi que **personne d'une autre équipe ne peut soumettre à votre place** et gaspiller vos 5 soumissions.

### Étape 6 — Soumettre via l'application

Une fois votre table sauvegardée, rendez-vous sur l'application du hackathon :

**[https://hackathon-sgdb-leaderboard-7474650176048310.aws.databricksapps.com](https://hackathon-sgdb-leaderboard-7474650176048310.aws.databricksapps.com)**

### Étapes de soumission

1. Cliquez sur l'onglet **"📤 Soumettre"**
2. Tapez le nom de votre équipe dans le champ *(sans le préfixe `equipe_`)*
   — ex : tapez `mon_equipe`, pas `equipe_mon_equipe`
3. Cliquez **"Calculer mon WAPE"** — votre score est calculé en direct
4. Vérifiez le score affiché, puis cliquez **"🏆 Soumettre officiellement"**
5. Votre score apparaît sur le leaderboard en temps réel

---

> ⚠️ **Limite : 5 soumissions par équipe.** Le score le plus récent est affiché sur le leaderboard.
> Utilisez "Calculer mon WAPE" autant que vous voulez pour vérifier votre score — cela ne consomme pas de soumission.

### Conseils clés

#### Données
- **~68 % de zéros** : prédire 0 pour les combinaisons agence × article rares est souvent optimal.
- **Saisonnalité forte** : la même semaine de l'année précédente est un signal puissant.
- **Enrichissez vos features** avec `donnees_agence` (région), `donnees_articles` (catégorie), `donnees_facturation` (prix).

#### Feature engineering — par où commencer

Les features les plus utiles sur ce type de données :

| Catégorie | Exemples |
|-----------|----------|
| **Lags temporels** | Quantité à N-1, N-2, N-4, N-8, N-26, N-52 semaines |
| **Même semaine N-1** | Quantité de la même semaine l'année précédente — signal saisonnier fort |
| **Moyennes mobiles** | Moyenne sur 4, 12, 26 semaines glissantes (avec shift pour éviter la fuite) |
| **Taux de zéros** | Part des semaines à zéro sur les 4, 12, 52 dernières semaines |
| **Stats agence** | Moyenne, médiane, écart-type des ventes par agence |
| **Stats article** | Moyenne, médiane, taux de zéros par article |
| **Stats paire agence × article** | Moyenne, max, count par paire — très prédictif |
| **Saisonnalité** | `sin(2π × semaine / 52)` et `cos(2π × semaine / 52)` |

> ⚠️ Attention à la **fuite de données** : les features basées sur l'historique doivent utiliser uniquement les données *antérieures* à la semaine prédite (utilisez `.shift(1)` avant les rolling).

#### Stratégie
- Validez toujours sur **2025-01 à 2025-26** avant de soumettre officiellement
- Un blend de plusieurs modèles bat généralement un modèle seul
- Consultez le notebook **`LightGBM_Example`** pour un pipeline complet : feature engineering, entraînement LightGBM, et génération des prédictions

---

**Bonne chance ! 🏆**